# Treinamento e avaliação dos modelos

## Imports 

In [34]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_validate
from sklearn.linear_model import LogisticRegression

## Lendo conjunto de dados

In [2]:
df = pd.read_csv('../data/raw/chatbot_medical_data.csv').drop('Unnamed: 0', axis=1)
df.head(10)

,pergunta,categoria,resposta
0,como marcar consulta?,consultas,"Para marcar consultas, acesse nosso portal ou ..."
1,preciso de um dermatologista urgente,dermatologia,"Dermatologistas atendem de segunda a sexta, da..."
2,dieta personalizada,nutricao,Nossa equipe de nutrição oferece atendimento p...
3,quero marcar com especialista do coração,cardiologia,Você pode agendar uma consulta com um cardiolo...
4,emagrecer com ajuda médica,nutricao,Nossa equipe de nutrição oferece atendimento p...
5,gripe forte,gripe,"Se estiver com sintomas de gripe, procure um c..."
6,quais convênios aceitam?,convenios,Trabalhamos com diversos convênios. Consulte a...
7,quero marcar com especialista do coração,cardiologia,Você pode agendar uma consulta com um cardiolo...
8,dor no osso,ortopedia,"A ortopedia atende casos de fraturas, dores mu..."
9,aceita plano de saúde?,convenios,Trabalhamos com diversos convênios. Consulte a...


## Pré-processamento

### Funções de pré-processamento de texto

In [3]:
# Função de pré-processamento do texto
def preprocess_text(text):
    # Transformando para minúsculas
    text = text.lower()
    
    # Remover símbolos e pontuação usando expressão regular
    text = re.sub(r'[^\w\s]', '', text)
    
    # Remover espaços extras
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

### Separar dados de treinamento e labels

In [4]:
X = df.pergunta
y = df.categoria

In [5]:
X.head(10)

0                       como marcar consulta?
1        preciso de um dermatologista urgente
2                         dieta personalizada
3    quero marcar com especialista do coração
4                  emagrecer com ajuda médica
5                                 gripe forte
6                    quais convênios aceitam?
7    quero marcar com especialista do coração
8                                 dor no osso
9                      aceita plano de saúde?
Name: pergunta, dtype: object

In [22]:
y.head(20)

0        consultas
1     dermatologia
2         nutricao
3      cardiologia
4         nutricao
5            gripe
6        convenios
7      cardiologia
8        ortopedia
9        convenios
10    dermatologia
11       consultas
12      neurologia
13    dermatologia
14    dermatologia
15        nutricao
16       ortopedia
17      neurologia
18    dermatologia
19        nutricao
Name: categoria, dtype: object

### Pré-processando perguntas

In [7]:
X_pre = X.apply(preprocess_text)
X_pre.head(10)

0                        como marcar consulta
1        preciso de um dermatologista urgente
2                         dieta personalizada
3    quero marcar com especialista do coração
4                  emagrecer com ajuda médica
5                                 gripe forte
6                     quais convênios aceitam
7    quero marcar com especialista do coração
8                                 dor no osso
9                       aceita plano de saúde
Name: pergunta, dtype: object

### Vetorizando o conjunto de dados X_pre para usar no treinamento dos modelos TFIDF

In [8]:
# Criando o vetorizador TF-IDF
tfidf = TfidfVectorizer()

# Transformando as perguntas em vetores TF-IDF
X = tfidf.fit_transform(df['pergunta'])

# Transformando os textos em vetores TF-IDF
X_tfidf = tfidf.fit_transform(X_pre)

In [9]:
# Convertendo o X_tfidf em um pandas dataframe
df_X_tfidf = pd.DataFrame(X_tfidf.toarray(), columns=tfidf.get_feature_names_out())

df_X_tfidf.head(10)

,aceita,aceitam,agendamento,agendar,ajuda,alergia,alimentação,ansioso,aí,cabeça,...,tem,to,tosse,trabalham,tristeza,um,uma,unimed,urgente,valor
0,0.000000,0.00000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0
1,0.000000,0.00000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.469751,0.0,0.0,0.521205,0.0
2,0.000000,0.00000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0
3,0.000000,0.00000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0
4,0.000000,0.00000,0.0,0.0,0.487264,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0
5,0.000000,0.00000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0
6,0.000000,0.57735,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0
7,0.000000,0.00000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0
8,0.000000,0.00000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0
9,0.590125,0.00000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0


### Convertendo as labels em valores inteiros

In [31]:
# Iniciando função de codificação
label_enconder =  LabelEncoder()

# Convertendo labels strings em labels int
y_encoded = label_enconder.fit_transform(y)

df_labels = pd.concat([y ,pd.DataFrame(y_encoded, columns=['numerica'])], axis=1)
df_labels.sample(30)

,categoria,numerica
8678,neurologia,5
8921,cardiologia,0
5681,ortopedia,7
185,nutricao,6
6886,neurologia,5
5872,neurologia,5
7886,cardiologia,0
2777,consultas,1
1088,dermatologia,3
9586,dermatologia,3


### Salvando o dado vetorizado e as labels

In [10]:
df_X_tfidf.to_csv('../data/processed/perguntas_vetorized_chatbot_dataset.csv')
df.categoria.to_csv('../data/processed/labels_chatbot_dataset.csv')

### Treinamento e avaliação dos modelos

#### Funções de cross-validation e resultados

In [ ]:
def cross_validation_evaluates(model, X, y, folds):

    # Definindo as métricas de avaliação
    scoring = ['precision_macro', 'recall_macro', 'f1_macro', 'accuracy']

    results = cross_validate(model, X_tfidf, y, cv=folds, scoring=scoring, return_train_score=False)

    # Exibindo os resultados médios
    print("Precision média:", results['test_precision_macro'].mean())
    print("Recall média:", results['test_recall_macro'].mean())
    print("F1-score média:", results['test_f1_macro'].mean())
    print("Acurácia média:", results['test_accuracy'].mean())

#### Regressão Logística

In [35]:
# Iniciando o modelo
model = LogisticRegression()

# Treinando e avaliando
cross_validation_evaluates(model, X_tfidf, y_encoded, 10)

Precision média: 1.0
Recall média: 1.0
F1-score média: 1.0
Acurácia média: 1.0


#### K-Nearest Neighbors